# Day 6 & Day 8 — US Flight Delay Project
**Dates:** 16 April 2026 (Day 6) · 21 April 2026 (Day 8)  
**Objective:** Deduplication · Airport Join · UDF · Haversine · Silver Delta · Caching  

---
> **Note:** This notebook continues from `US-Project-File3.ipynb`.  
> It re-reads Bronze Delta and re-applies all File3 transformations  
> so it is fully self-contained.

## Step 0 — Re-read Bronze Delta & Replay Transformations

In [0]:
# 0.1  Read Bronze Delta
df = spark.read.format("delta").load("s3://airlines-bucket-07860/Bronze_data/flight")
print(f"Bronze rows : {df.count():,}")
print(f"Bronze cols : {len(df.columns)}")

Bronze rows : 2,797,629
Bronze cols : 51


In [0]:
# 0.2  Replay all File3 (Day 5/6) Silver transformations
from pyspark.sql.functions import (
    col, when, floor, concat, lpad, lit, upper,
    year, month, quarter, dayofweek, to_date
)

# --- CRS time → HH:MM string (null-safe, 2400 → 00:00) ---
df = df \
    .withColumn(
        "crs_dep_time_std",
        when(col("CRSDepTime").isNull(), None)
        .when(col("CRSDepTime") == 2400, lit("00:00"))
        .otherwise(
            concat(
                lpad(floor(col("CRSDepTime") / 100).cast("int"), 2, "0"),
                lit(":"),
                lpad((col("CRSDepTime") % 100).cast("int"), 2, "0")
            )
        )
    ) \
    .withColumn(
        "crs_arr_time_std",
        when(col("CRSArrTime").isNull(), None)
        .when(col("CRSArrTime") == 2400, lit("00:00"))
        .otherwise(
            concat(
                lpad(floor(col("CRSArrTime") / 100).cast("int"), 2, "0"),
                lit(":"),
                lpad((col("CRSArrTime") % 100).cast("int"), 2, "0")
            )
        )
    )

# --- CRS time → total minutes (null-safe, 2400 → 0) ---
df = df \
    .withColumn(
        "crs_dep_minutes",
        when(col("CRSDepTime").isNull(), None)
        .when(col("CRSDepTime") == 2400, lit(0))
        .otherwise(floor(col("CRSDepTime") / 100) * 60 + (col("CRSDepTime") % 100))
    ) \
    .withColumn(
        "crs_arr_minutes",
        when(col("CRSArrTime").isNull(), None)
        .when(col("CRSArrTime") == 2400, lit(0))
        .otherwise(floor(col("CRSArrTime") / 100) * 60 + (col("CRSArrTime") % 100))
    )

# --- Boolean flags ---
df = df \
    .withColumn("is_cancelled",        when(col("Cancelled") == 1, True).otherwise(False)) \
    .withColumn("is_diverted",          when(col("Diverted") == 1, True).otherwise(False)) \
    .withColumn("is_departure_delayed", when(col("DepDel15") == 1, True).otherwise(False)) \
    .withColumn("is_arrival_delayed",   when(col("ArrivalDelayGroups") > 0, True).otherwise(False))

# --- Arrival delay category ---
df = df.withColumn(
    "arrival_delay_category",
    when(col("Cancelled") == 1, "Cancelled")
    .when(col("ArrDelayMinutes") < 0, "Early")
    .when((col("ArrDelayMinutes") >= 0)  & (col("ArrDelayMinutes") <= 14),  "On Time")
    .when((col("ArrDelayMinutes") >= 15) & (col("ArrDelayMinutes") <= 44),  "Minor Delay")
    .when((col("ArrDelayMinutes") >= 45) & (col("ArrDelayMinutes") <= 120), "Major Delay")
    .when(col("ArrDelayMinutes") > 120, "Severe Delay")
    .otherwise(None)
)

# --- Date feature extraction ---
df = df \
    .withColumn("FlightDate",     to_date(col("FlightDate"))) \
    .withColumn("flight_year",    year(col("FlightDate"))) \
    .withColumn("flight_month",   month(col("FlightDate"))) \
    .withColumn("flight_quarter", quarter(col("FlightDate"))) \
    .withColumn("day_of_week",    dayofweek(col("FlightDate")))

# --- Uppercase code columns ---
df = df \
    .withColumn("airline_code",    upper(col("IATA_CODE_Reporting_Airline"))) \
    .withColumn("origin_code",     upper(col("Origin"))) \
    .withColumn("destination_code",upper(col("Dest")))

# --- Cancellation reason ---
df = df.withColumn(
    "cancellation_reason",
    when(col("CancellationCode") == "A", "Carrier")
    .when(col("CancellationCode") == "B", "Weather")
    .when(col("CancellationCode") == "C", "National_Security")
    .when(col("CancellationCode") == "D", "Security_Concern")
    .otherwise(None)
)

print(f"Columns after replay: {len(df.columns)}")
print("Replay complete.")

Columns after replay: 68
Replay complete.


---
# Day 6 — 16 April 2026
**Topics:** Duplicate detection with window functions · load_timestamp · Airport broadcast join

## 6.1 — Duplicate Detection with Window Functions

In [0]:
# Add load_timestamp before dedup so we can order by ingestion time
from pyspark.sql.functions import current_timestamp
df = df.withColumn("load_timestamp", current_timestamp())

In [0]:
# Detect duplicates:
# Partition by the 5 flight-identity keys.
# Order by load_timestamp DESC so row_num=1 = most recently ingested record.
from pyspark.sql.window   import Window
from pyspark.sql.functions import row_number

dedup_window = Window.partitionBy(
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "Dest"
).orderBy(col("load_timestamp").desc())
df = df.withColumn("row_num", row_number().over(dedup_window))
total_before  = df.count()
duplicate_cnt = df.filter(col("row_num") > 1).count()
print(f"Total rows          : {total_before:,}")
print(f"Duplicate rows      : {duplicate_cnt:,}")
print(f"Unique rows to keep : {total_before - duplicate_cnt:,}")

Total rows          : 2,797,629
Duplicate rows      : 0
Unique rows to keep : 2,797,629


In [0]:
# Keep only the first occurrence per flight (row_num == 1)
df = df.filter(col("row_num") == 1).drop("row_num")
print(f"Rows after deduplication: {df.count():,}")

Rows after deduplication: 2,797,629


## 6.2 — Airport Delta Table: Broadcast Join with Flight Data

In [0]:
# Read US airport Delta table (created in File3 / Day 4)
df_airport = spark.read.format("delta") \
    .load("s3://airlines-bucket-07860/Airport_data/delta/")
print(f"Airport rows (US): {df_airport.count():,}")
df_airport.printSchema()

Airport rows (US): 13,578
root
 |-- city: string (nullable = true)
 |-- code: string (nullable = true)
 |-- country: string (nullable = true)
 |-- elevation: long (nullable = true)
 |-- iata: string (nullable = true)
 |-- icao: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- tz: string (nullable = true)



In [0]:
# Prepare two views of airport data: one for Origin, one for Destination
# Rename columns to avoid ambiguity after join.
from pyspark.sql.functions import broadcast
df_origin_airport = df_airport.select(
    col("iata").alias("origin_iata"),
    col("name").alias("origin_airport_name"),
    col("city").alias("origin_airport_city"),
    col("state").alias("origin_airport_state"),
    col("lat").alias("origin_lat"),
    col("lon").alias("origin_lon"),
    col("tz").alias("origin_timezone")
)
df_dest_airport = df_airport.select(
    col("iata").alias("dest_iata"),
    col("name").alias("dest_airport_name"),
    col("city").alias("dest_airport_city"),
    col("state").alias("dest_airport_state"),
    col("lat").alias("dest_lat"),
    col("lon").alias("dest_lon"),
    col("tz").alias("dest_timezone")
)
print("Airport alias DataFrames created.")

Airport alias DataFrames created.


In [0]:
# Broadcast join — Origin airport
# broadcast() sends the small airport DataFrame to every executor,
# avoiding a shuffle of the large flight DataFrame.
df_silver = df.join(
    broadcast(df_origin_airport),
    df.origin_code == df_origin_airport.origin_iata,
    how="left"
).drop("origin_iata")

print(f"After origin join  : {df_silver.count():,} rows")
print(f"Column count       : {len(df_silver.columns)}")

After origin join  : 2,797,629 rows
Column count       : 75


In [0]:
# Broadcast join — Destination airport
df_silver = df_silver.join(
    broadcast(df_dest_airport),
    df_silver.destination_code == df_dest_airport.dest_iata,
    how="left"
).drop("dest_iata")
print(f"After dest join    : {df_silver.count():,} rows")
print(f"Column count       : {len(df_silver.columns)}")

After dest join    : 2,797,629 rows
Column count       : 81


In [0]:
# display(df_silver)    

In [0]:
# Validate: confirm airport details enriched correctly
df_silver.select(
    "FlightDate",
    "origin_code",  "origin_airport_name",  "origin_lat",  "origin_lon",
    "destination_code", "dest_airport_name", "dest_lat",    "dest_lon"
).show(5, truncate=False)

+----------+-----------+------------------------------------------------+-------------+--------------+----------------+----------------------------------------------------------+-------------+--------------+
|FlightDate|origin_code|origin_airport_name                             |origin_lat   |origin_lon    |destination_code|dest_airport_name                                         |dest_lat     |dest_lon      |
+----------+-----------+------------------------------------------------+-------------+--------------+----------------+----------------------------------------------------------+-------------+--------------+
|2021-01-01|GRB        |Austin Straubel International Airport           |44.4850997925|-88.1296005249|MSP             |Minneapolis-St Paul International/Wold-Chamberlain Airport|44.8819999695|-93.2218017578|
|2021-01-01|ATL        |Hartsfield Jackson Atlanta International Airport|33.6366996765|-84.4281005859|LFT             |Lafayette Regional Airport                       

---
# Day 8 — 21 April 2026
**Topics:** UDF (flight number extraction) · Haversine distance · Silver Delta save · Caching

## 8.1 — UDF: Extract 1st, 4th, and 7th Characters from Flight Number

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types     import StringType

def extract_flight_chars(flight_num):
    """
    Extract the 1st, 4th, and 7th characters (1-indexed) from the flight
    number string and return them concatenated.
    Returns None if input is None.
    If the string is shorter than the target position, that character is skipped.

    Examples:
        '1234567'  -> '147'
        'AA123'    -> 'A1'  (no 7th char)
        None       -> None
    """
    if flight_num is None:
        return None
    s = str(flight_num)
    result = ""
    for idx in [0, 3, 6]:   # 0-based indices for positions 1, 4, 7
        if idx < len(s):
            result += s[idx]
    return result if result else None

# Register as PySpark UDF
extract_flight_chars_udf = udf(extract_flight_chars, StringType())

# Register for Spark SQL as well
spark.udf.register("extract_flight_chars", extract_flight_chars, StringType())

print("UDF registered.")

# Quick local test before applying to DataFrame
test_cases = ["1234567", "AA123", "WN2345", None, "1"]
for t in test_cases:
    print(f"  extract_flight_chars({t!r:12}) -> {extract_flight_chars(t)!r}")

UDF registered.
  extract_flight_chars('1234567'   ) -> '147'
  extract_flight_chars('AA123'     ) -> 'A2'
  extract_flight_chars('WN2345'    ) -> 'W3'
  extract_flight_chars(None        ) -> None
  extract_flight_chars('1'         ) -> '1'


In [0]:
# Apply UDF to Silver DataFrame
df_silver = df_silver.withColumn(
    "flight_num_extract",
    extract_flight_chars_udf(col("Flight_Number_Reporting_Airline"))
)

# Validate
df_silver.select(
    "Flight_Number_Reporting_Airline",
    "flight_num_extract"
).show(10, truncate=False)

+-------------------------------+------------------+
|Flight_Number_Reporting_Airline|flight_num_extract|
+-------------------------------+------------------+
|4643                           |43                |
|4683                           |43                |
|4686                           |46                |
|4718                           |48                |
|4733                           |43                |
|4734                           |44                |
|4735                           |45                |
|4743                           |43                |
|4749                           |49                |
|4763                           |43                |
+-------------------------------+------------------+
only showing top 10 rows


## 8.2 — Haversine Distance Function
Computes the great-circle distance (in km) between origin and destination airports  
using their latitude/longitude coordinates enriched by the airport join.

In [0]:
# Haversine formula implemented with native Spark functions
# (no Python UDF) for maximum performance on distributed data.
#
# Formula:
#   a = sin^2(delta_lat/2) + cos(lat1)*cos(lat2)*sin^2(delta_lon/2)
#   c = 2 * atan2( sqrt(a), sqrt(1 - a) )
#   d = R * c      (R = 6371 km)

from pyspark.sql.functions import (
    radians, sin, cos, sqrt, atan2, lit
)
from pyspark.sql.functions import pow as spark_pow

def add_haversine_distance(
        df,
        lat1_col="origin_lat",
        lon1_col="origin_lon",
        lat2_col="dest_lat",
        lon2_col="dest_lon",
        result_col="haversine_distance_km"):
    """
    Add column result_col with the Haversine great-circle distance in km.
    Returns NULL when any coordinate is NULL.
    """
    R = 6371.0    # Earth radius in km

    lat1 = radians(col(lat1_col))
    lat2 = radians(col(lat2_col))
    lon1 = radians(col(lon1_col))
    lon2 = radians(col(lon2_col))

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = spark_pow(sin(dlat / 2), lit(2)) \
        + cos(lat1) * cos(lat2) * spark_pow(sin(dlon / 2), lit(2))

    c = lit(2) * atan2(sqrt(a), sqrt(lit(1.0) - a))

    return df.withColumn(result_col, lit(R) * c)


df_silver = add_haversine_distance(df_silver)

# Validate
df_silver.select(
    "origin_code",  "origin_lat",  "origin_lon",
    "destination_code", "dest_lat", "dest_lon",
    "haversine_distance_km"
).show(10, truncate=False)

+-----------+-------------+--------------+----------------+-------------+--------------+---------------------+
|origin_code|origin_lat   |origin_lon    |destination_code|dest_lat     |dest_lon      |haversine_distance_km|
+-----------+-------------+--------------+----------------+-------------+--------------+---------------------+
|GRB        |44.4850997925|-88.1296005249|MSP             |44.8819999695|-93.2218017578|404.9316543514658    |
|ATL        |33.6366996765|-84.4281005859|LFT             |30.20529938  |-91.98760223  |808.7514193578977    |
|GRR        |42.88079834  |-85.52279663  |DTW             |42.2123985291|-83.3534011841|192.62514774191874   |
|GRR        |42.88079834  |-85.52279663  |DTW             |42.2123985291|-83.3534011841|192.62514774191874   |
|DTW        |42.2123985291|-83.3534011841|MKE             |42.9472007751|-87.8965988159|380.7977917891175    |
|OKC        |35.3931007385|-97.6007003784|MSP             |44.8819999695|-93.2218017578|1118.3988181524333   |
|

In [0]:
# Compare Haversine (km) vs reported Distance (miles)
from pyspark.sql.functions import round as spark_round

df_silver.select(
    "origin_code",
    "destination_code",
    "Distance",
    spark_round("haversine_distance_km", 2).alias("haversine_km")
).filter(col("haversine_distance_km").isNotNull()) \
 .show(15, truncate=False)


+-----------+----------------+--------+------------+
|origin_code|destination_code|Distance|haversine_km|
+-----------+----------------+--------+------------+
|GRB        |MSP             |252.0   |404.93      |
|ATL        |LFT             |503.0   |808.75      |
|GRR        |DTW             |120.0   |192.63      |
|GRR        |DTW             |120.0   |192.63      |
|DTW        |MKE             |237.0   |380.8       |
|OKC        |MSP             |694.0   |1118.4      |
|MSP        |CMH             |626.0   |1005.52     |
|MSP        |DFW             |852.0   |1372.58     |
|MSP        |RST             |76.0    |122.52      |
|MCO        |RDU             |534.0   |861.45      |
|ATL        |HOU             |696.0   |1118.18     |
|ICT        |ATL             |782.0   |1255.68     |
|DAY        |DTW             |166.0   |266.92      |
|JFK        |RDU             |427.0   |686.5       |
|MSP        |MOT             |449.0   |720.88      |
+-----------+----------------+--------+-------

## 8.3 — Save Silver DataFrame as Delta Table

In [0]:
# Define Silver path
# s3://airlines-bucket-07860/Airport_data/silver_data/flight
SILVER_PATH = "s3://airlines-bucket-07860/silver_data/flight"
print(f"Silver Delta path : {SILVER_PATH}")
print(f"Rows to write     : {df_silver.count():,}")
print(f"Columns to write  : {len(df_silver.columns)}")

Silver Delta path : s3://airlines-bucket-07860/silver_data/flight
Rows to write     : 2,797,629
Columns to write  : 83


In [0]:
# Write Silver Delta
# - Partition by year + month (same as Bronze) for efficient time-range queries
# - delta.dataSkippingNumIndexedCols = -1  indexes ALL columns,
#   enabling data-skipping on any predicate during selective reads

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.dataSkippingNumIndexedCols", "-1") \
    .partitionBy("year", "month") \
    .save(SILVER_PATH)

print("Silver Delta table written successfully.")

Silver Delta table written successfully.


In [0]:
%sql
USE Flight_Analytics;
CREATE TABLE IF NOT EXISTS silver_flights
USING DELTA
LOCATION 's3://airlines-bucket-07860/silver_data/flight';
SELECT COUNT(*) AS row_count FROM Flight_Analytics.silver_flights;

row_count
2797629


## 8.4 — Phase 3: Read Silver Delta · Caching & Performance Comparison

In [0]:
# import time

# # Read Silver Delta fresh (lazy — no cache)
# df_read = spark.read.format("delta").load(SILVER_PATH)

# # Measure count WITHOUT cache
# t0 = time.time()
# count_no_cache = df_read.count()
# t1 = time.time()
# time_no_cache = round(t1 - t0, 3)

# print(f"Row count (no cache) : {count_no_cache:,}")
# print(f"Time  (no cache)     : {time_no_cache} sec")

In [0]:
# # Cache the DataFrame in memory
# # .cache() = .persist(StorageLevel.MEMORY_AND_DISK)
# # First action after .cache() fills the cache;
# # subsequent actions read from memory instead of S3.

# df_read.createOrReplaceTempView("temp_df") # Use of TempView is mandate here because .cache is not supported on a DataFrame as serverless compute/
# df_read.count()    # triggers cache population

# print("Cache populated.")

In [0]:
# # Measure count WITH cache
# t2 = time.time()
# count_with_cache = df_read.count()
# t3 = time.time()
# time_with_cache = round(t3 - t2, 3)

# print(f"Row count (cached)   : {count_with_cache:,}")
# print(f"Time  (cached)       : {time_with_cache} sec")

In [0]:
# # Performance comparison summary
# speedup = round(time_no_cache / time_with_cache, 2) if time_with_cache > 0 else 'N/A'
# print()
# print('  CACHING PERFORMANCE COMPARISON')
# print(f'  Without cache : {time_no_cache:>8.3f} sec')
# print(f'  With cache    : {time_with_cache:>8.3f} sec')
# print(f'  Speedup       : {speedup}x')

In [0]:
# Release cache once done to free executor memory
# df_read.write.mode("overwrite").saveAsTable("temp_df_mat")

In [0]:
# %sql
# SELECT * FROM temp_df_mat;

# Day 9 — US Flight Delay Project
**Date:** 23 April 2026  
**Objective:** Aggregation & Analytics on the Silver Delta Table  

---
> **Note:** This notebook reads directly from the Silver Delta table.  
> No transformations are replayed — the Silver table is the clean, enriched source of truth.

**Key column-name notes (task terminology → actual silver column):**
| Task name | Actual silver column | Type |
|-----------|----------------------|------|
| `reporting_airline` | `Reporting_Airline` | StringType |
| `is_arr_delayed` | `is_arrival_delayed` | BooleanType |
| `arr_delay_minutes` | `ArrDelayMinutes` | FloatType |
| `is_cancelled` | `is_cancelled` | BooleanType |
| `is_diverted` | `is_diverted` | BooleanType |
| `carrier_delay` | `CarrierDelay` | FloatType |
| `weather_delay` | `WeatherDelay` | FloatType |
| `security_delay` | `SecurityDelay` | FloatType |
| `late_aircraft_delay` | `LateAircraftDelay` | FloatType |
| `distance` | `Distance` | FloatType |

> **Important:** `is_cancelled`, `is_arrival_delayed`, and `is_diverted` are **BooleanType**  
> (created with `when(..., True).otherwise(False)` in File3/File4).  
> For `sum()` and `count()` aggregations they must be cast to `IntegerType` first.  
> All casts are handled inside the aggregation expressions below.

---
## Step 1 — Read Silver Delta Table
Task item 1: *Create a DataFrame named `df_silver` by reading the Delta table from the Silver path.*

In [0]:
# 1. Read Silver Delta table into df_silver
SILVER_PATH = "s3://airlines-bucket-07860/silver_data/flight"
df_silver = spark.read.format("delta").load(SILVER_PATH)
print(f"Silver rows : {df_silver.count():,}")
print(f"Silver cols : {len(df_silver.columns)}")
# df_silver.printSchema()

Silver rows : 2,797,629
Silver cols : 83


---
## Step 2 — Total Flights Grouped by flight_year
Task item 2 & 3: *Group by `flight_year`, count total flights.*

In [0]:
from pyspark.sql.functions import col, count
# 2. Group by flight_year → total number of flights
df_by_year = df_silver \
    .groupBy("flight_year") \
    .agg(
        count("*").alias("total_flights")
    ) \
    .orderBy("flight_year")
# display(df_by_year)

---
## Step 3 — Total Flights Grouped by flight_year and flight_month
Task item 4: *Group by `flight_year`, `flight_month` → total flights per year-month.*

In [0]:
# 3. Group by flight_year + flight_month → total flights
df_by_year_month = df_silver \
    .groupBy("flight_year", "flight_month") \
    .agg(
        count("*").alias("total_flights")
    ) \
    .orderBy("flight_year", "flight_month")
# display(df_by_year_month)

---
## Step 4 — Cancelled Flights Grouped by flight_year and flight_month
Task item 5: *`sum(is_cancelled)` grouped by `flight_year`, `flight_month`.*  
> `is_cancelled` is **BooleanType** — casting to `IntegerType` before `sum()` is mandatory.

In [0]:
from pyspark.sql.functions import sum as spark_sum
# 4. Cancelled flights per year-month
# is_cancelled is BooleanType → cast to int so sum() counts True as 1
df_cancelled_ym = df_silver \
    .groupBy("flight_year", "flight_month") \
    .agg(
        count("*").alias("total_flights"),
        spark_sum(col("is_cancelled").cast("int")).alias("cancelled_flights")
    ) \
    .orderBy("flight_year", "flight_month")
# display(df_cancelled_ym)

---
## Step 5 — Group By flight_year, flight_month, airline_code, Reporting_Airline
Task items 6 & 7: *Calculate total flights, delayed flights, cancelled flights, diverted flights.*

In [0]:
# 5. Group by 4 columns — base flight counts
# GROUP BY keys:
#   flight_year, flight_month, airline_code, Reporting_Airline
#
# is_arrival_delayed = actual column name for task's 'is_arr_delayed'
# is_cancelled / is_diverted = BooleanType → cast to int for sum()

GROUP_COLS = ["flight_year", "flight_month", "airline_code", "Reporting_Airline"]
df_base = df_silver \
    .groupBy(*GROUP_COLS) \
    .agg(
        count("*").alias("total_number_of_flights"),
        spark_sum(col("is_arrival_delayed").cast("int")).alias("delayed_flights"),
        spark_sum(col("is_cancelled").cast("int")).alias("total_flights_cancelled"),
        spark_sum(col("is_diverted").cast("int")).alias("diverted_flights")
    ) \
    .orderBy("flight_year", "flight_month", "airline_code")
# display(df_base)

---
## Step 6 — Average & Median Arrival Delay (Non-Cancelled Flights Only)
Task items 8 & 9:  
- `avg(ArrDelayMinutes)` → `avg_arrival_delay_minutes`  
- `percentile_approx(ArrDelayMinutes, 0.5)` → `median_arrival_delay_minutes`  
- **Condition:** only rows where `is_cancelled = False` (i.e. non-cancelled flights)

In [0]:
from pyspark.sql.functions import avg, percentile_approx
# 6. Average and Median arrival delay — non-cancelled flights only
# Filter first: is_cancelled == False excludes cancelled flights
# ArrDelayMinutes is the actual column (task calls it arr_delay_minutes)

df_delay_stats = df_silver \
    .filter(col("is_cancelled") == False) \
    .groupBy(*GROUP_COLS) \
    .agg(
        count("*").alias("non_cancelled_flights"),
        avg(col("ArrDelayMinutes")).alias("avg_arrival_delay_minutes"),
        percentile_approx(col("ArrDelayMinutes"), 0.5)
            .alias("median_arrival_delay_minutes")
    ) \
    .orderBy("flight_year", "flight_month", "airline_code")
# display(df_delay_stats)

---
## Step 7 — On-Time Flights Count & Percentage
Task item 10 & 11:  
- `on_time_flights` → count where `is_cancelled = False AND is_arrival_delayed = False`  
- `on_time_flight_percentage` → `(on_time_flights / total_flights) * 100`

In [0]:
from pyspark.sql.functions import (
    round as spark_round,
    when,
    lit,
    col,
    count,
    sum as spark_sum
)
# 7. On-time flights and on-time percentage
# A flight is on-time when it is NOT cancelled AND NOT arrival-delayed.
df_ontime = df_silver \
    .groupBy(*GROUP_COLS) \
    .agg(
        count("*").alias("total_number_of_flights"),
        spark_sum(
            when(
                (col("is_cancelled") == False) & (col("is_arrival_delayed") == False),
                lit(1)
            ).otherwise(lit(0))
        ).alias("on_time_flights")
    ) \
    .withColumn(
        "on_time_flight_percentage",
        spark_round(
            (col("on_time_flights") / col("total_number_of_flights")) * 100,
            2
        )
    ) \
    .orderBy("flight_year", "flight_month", "airline_code")
# display(df_ontime)

---
## Step 8 — Cancelled Flight Percentage
Task item 12: `(cancelled_flights / total_flights) * 100`

In [0]:
# 8. Cancelled flight percentage
from pyspark.sql.functions import lit
df_cancelled_pct = df_silver \
    .groupBy(*GROUP_COLS) \
    .agg(
        count("*").alias("total_number_of_flights"),
        spark_sum(col("is_cancelled").cast("int")).alias("cancelled_flights")
    ) \
    .withColumn(
        "cancelled_flight_percentage",
        spark_round(
            (col("cancelled_flights") / col("total_number_of_flights")) * 100,
            2
        )
    ) \
    .orderBy("flight_year", "flight_month", "airline_code")
# display(df_cancelled_pct)

---
## Step 9 — Average Delay by Type (Carrier, Weather, Security, Late Aircraft)
Task items 13–16:  
- `avg(CarrierDelay)` → `avg_carrier_delay`  
- `avg(WeatherDelay)` → `avg_weather_delay`  
- `avg(SecurityDelay)` → `avg_security_delay`  
- `avg(LateAircraftDelay)` → `avg_late_aircraft_delay`  
> These delay columns contain values only when a flight is delayed by that cause;  
> `avg()` naturally ignores NULLs, so cancelled/on-time flights don't skew the mean.

In [0]:
# 9. Average delay by type
# Column name note: task says 'aircraft_delay' as alternative → actual col is 'LateAircraftDelay'
df_delay_types = df_silver \
    .groupBy(*GROUP_COLS) \
    .agg(
        spark_round(avg(col("CarrierDelay")),      2).alias("avg_carrier_delay"),
        spark_round(avg(col("WeatherDelay")),      2).alias("avg_weather_delay"),
        spark_round(avg(col("SecurityDelay")),     2).alias("avg_security_delay"),
        spark_round(avg(col("LateAircraftDelay")), 2).alias("avg_late_aircraft_delay")
    ) \
    .orderBy("flight_year", "flight_month", "airline_code")
# display(df_delay_types)

---
## Step 10 — Distance Metrics
Task item 17:  
- `avg(Distance)` → `avg_distance_travelled`  
- `sum(Distance)` → `total_distance_travelled`

In [0]:
# 10. Distance metrics — average and total distance
# Column name: 'Distance' (exactly as in the 37-column silver schema)
df_distance = df_silver \
    .groupBy(*GROUP_COLS) \
    .agg(
        spark_round(avg(col("Distance")),              2).alias("avg_distance_travelled"),
        spark_round(spark_sum(col("Distance")),        0).alias("total_distance_travelled")
    ) \
    .orderBy("flight_year", "flight_month", "airline_code")
# display(df_distance)

In [0]:
# Combine all metrics into a single DataFrame (gold layer)
gold_airline_kpi= df_base \
    .join(df_delay_stats, on=GROUP_COLS, how="left") \
    .join(df_ontime.drop("total_number_of_flights"), on=GROUP_COLS, how="left") \
    .join(df_cancelled_pct.drop("total_number_of_flights"), on=GROUP_COLS, how="left") \
    .join(df_delay_types, on=GROUP_COLS, how="left") \
    .join(df_distance, on=GROUP_COLS, how="left")

# display(gold_airline_kpi)

# First Gold Kpi 
 ##"gold_airline_kpi"

In [0]:
# Create monthly_rank
# Partition by flight_year, flight_month
# Order by total_number_of_flights DESC
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank
rank_window = Window.partitionBy("flight_year", "flight_month") \
                    .orderBy(col("total_number_of_flights").desc())

gold_airline_kpi = gold_airline_kpi.withColumn(
    "monthly_rank",
    dense_rank().over(rank_window)
)

display(gold_airline_kpi)



# GOLD_PATH = "s3://airlines-bucket-07860/gold_data/flight"

gold_airline_kpi.write.format("delta").mode("overwrite").saveAsTable("gold_airline_kpi")


flight_year,flight_month,airline_code,Reporting_Airline,total_number_of_flights,delayed_flights,total_flights_cancelled,diverted_flights,non_cancelled_flights,avg_arrival_delay_minutes,median_arrival_delay_minutes,on_time_flights,on_time_flight_percentage,cancelled_flights,cancelled_flight_percentage,avg_carrier_delay,avg_weather_delay,avg_security_delay,avg_late_aircraft_delay,avg_distance_travelled,total_distance_travelled,monthly_rank
2021,1,WN,WN,61307,4883,677,43,60630,4.2302144024295645,0.0,55747,90.93,677,1.1,16.5,0.46,0.1,14.09,756.62,4.6386341E7,1
2021,1,OO,OO,52000,5190,588,165,51412,9.583526840595548,0.0,46222,88.89,588,1.13,50.9,16.99,0.27,14.74,538.96,2.8025745E7,2
2021,1,DL,DL,48239,4347,92,47,48147,6.782619542619543,0.0,43800,90.8,92,0.19,31.71,2.23,0.15,13.76,1005.83,4.8520206E7,3
2021,1,AA,AA,38226,3927,258,62,37968,8.18221389753601,0.0,34041,89.05,258,0.67,29.5,6.23,0.28,16.12,1033.12,3.9491916E7,4
2021,1,UA,UA,23960,1988,162,36,23798,6.034677215722582,0.0,21810,91.03,162,0.68,20.42,5.19,0.0,17.19,1181.56,2.8310188E7,5
2021,1,YX,YX,21720,1839,271,28,21449,6.278044909201251,0.0,19610,90.29,271,1.25,22.68,6.37,0.21,15.71,551.63,1.1981381E7,6
2021,1,9E,9E,19501,1384,23,25,19478,6.334652752788773,0.0,18094,92.78,23,0.12,30.32,17.09,0.06,20.42,385.92,7525796.0,7
2021,1,MQ,MQ,17665,2634,331,44,17334,10.104453441295547,0.0,14700,83.22,331,1.87,11.16,9.62,0.24,13.79,508.44,8981659.0,8
2021,1,OH,OH,12615,1464,373,16,12242,9.439964011123834,0.0,10778,85.44,373,2.96,28.52,3.76,0.18,22.67,405.64,5117146.0,9
2021,1,YV,YV,11232,1302,172,16,11060,9.50642883013401,0.0,9758,86.88,172,1.53,26.27,11.03,0.08,15.73,679.38,7630826.0,10


---
# Gold Layer Based on Route 
## "gold_route_kpi"

In [0]:

from pyspark.sql.functions import col, when, count, sum, avg, round, concat_ws, countDistinct


# Step 1: Route column create karo
# route = origin_code + "-" + destination_code

df_route = df_silver.withColumn(
    "route",
    concat_ws("-", col("origin_code"), col("destination_code"))
)


In [0]:

# Step 2: On-time flight flag create karo
# Condition:
# 1 = flight cancelled nahi honi chahiye
# 2 = flight arrival delayed bhi nahi honi chahiye
# On-time = valid flight + no arrival delay
df_route = df_route.withColumn(
    "on_time_flight",
    when(
        (col("is_cancelled") == False) & (col("is_arrival_delayed") == False),
        1
    ).otherwise(0)
)

In [0]:



# Step 3: Group by route level KPI calculate karo
# Group by:
# - flight_year
# - route
# - origin_code
# - destination_code
gold_route_kpi = df_route.groupBy(
    "flight_year",
    "route",
    "origin_code",
    "destination_code"
).agg(
    # 1. Total number of flights on that route
    count("*").alias("number_of_flights"),

    # 2. Average arrival delay (only non-cancelled flights)
    round(avg(when(col("is_cancelled") == False, col("ArrDelayMinutes"))
        ),2).alias("avg_arrival_delay"),

    # 3. Average distance travelled on that route
    round(avg("Distance"),2).alias("avg_distance_travelled"),

    # 4. Total delayed flights on that route
    sum(
        when(col("is_arrival_delayed") == True, 1).otherwise(0)
    ).alias("total_delayed_flights"),

    # 5. Total on-time flights
    sum("on_time_flight").alias("total_on_time_flights"),

    # 6. Number of unique airlines on that route
    countDistinct("airline_code").alias("number_of_airlines_on_route")
)

# Step 4: On-time flight percentage calculate karo
# Formula:
# (total_on_time_flights / number_of_flights) * 100

gold_route_kpi = gold_route_kpi.withColumn(
    "on_time_percentage_airline_percentage",
    round((col("total_on_time_flights") / col("number_of_flights")) * 100,2)
)

# Step 6: Final display
# display(gold_route_kpi)

# Step 7: Save as Delta Table
# Table name = gold_route_kpi
gold_route_kpi.write.format("delta").mode("overwrite").saveAsTable("gold_route_kpi")


## Gold Layer Based On Departure
# "gold_airport_departure_kpi"

In [0]:
from pyspark.sql.functions import col
df_join=df_silver.join(df_airport ,
                          df_silver["Origin"]==df_airport ["iata"],"inner")

In [0]:
# display(df_join)
# print(df_join.columns)

In [0]:
from pyspark.sql.functions import col, when, count, sum, avg, round, concat_ws


# Step 1: Base dataframe create karo
# Departure on-time flag create karo
# Condition:
# flight cancelled nahi honi chahiye
# aur departure delayed bhi nahi honi chahiye
df_departure = df_join.withColumn(
    "departure_on_time_flight",
    when(
        (col("is_cancelled") == False) & (col("is_departure_delayed") == False),
        1
    ).otherwise(0)
)


In [0]:
# df_departure.printSchema()

In [0]:
from pyspark.sql.functions import col, count, sum, avg, when, round

df_gold_airport_departure_kpi = df_departure.groupBy(
    "flight_year",
    "flight_month",
    "origin_code",
    "origin_airport_name",
    "origin_airport_city",
    "origin_airport_state",
    "origin_lon",
    "origin_lat"
).agg(

    # 1. Total departures
    count("*").alias("total_departure"),

    # 2. Total cancelled departures
    sum(
        when(col("is_cancelled") == True, 1).otherwise(0)
    ).alias("total_cancelled_departure"),

    # 3. Average departure delay (only non-cancelled flights)
    round(
        avg(
            when(col("is_cancelled") == False, col("DepDelayMinutes"))
        ),
        2
    ).alias("avg_departure_delay_minutes"),

    # 4. Total on-time departures (NOT delayed)
    sum(
        when(col("is_departure_delayed") == False, 1).otherwise(0)
    ).alias("total_on_time_departure"),

    # 5. Average route distance
    round(
        avg(col("Distance")),
        2
    ).alias("avg_route_distance"),

    # 6. Number of flights operating
    count("*").alias("number_of_flights_operating"),

    # 7. Average airtime (non-cancelled flights)
    round(
        avg(
            when(col("is_cancelled") == False, col("AirTime"))
        ),
        2
    ).alias("avg_airtime")

)

In [0]:
# Step 3: Departure on-time percentage calculate karo
# Formula:
# (total_on_time_departure / total_departure) * 100

df_gold_airport_departure_kpi = df_gold_airport_departure_kpi.withColumn(
    "departure_on_time_percentage",
    round((col("total_on_time_departure") / col("total_departure")) * 100,2)
)


In [0]:
# Step 5: Helper column drop karo
# total_on_time_departure sirf percentage ke liye use hui thi

df_gold_departure_kpi = df_gold_airport_departure_kpi.drop("total_on_time_departure")
# Step 6: Final display
# display(df_gold_departure_kpi)
# Step 7: Save as Delta Table
# Table name = gold_airport_departure_kpi
df_gold_departure_kpi.write.format("delta").mode("overwrite").saveAsTable("gold_airport_departure_kpi")

# Gold Layer Based on Delay Cause
## "df_delay_cause_table"

In [0]:
from pyspark.sql.functions import col, sum, round


# Step 1: Filter only valid delayed flights
# Conditions:
# 1. Flight cancelled nahi honi chahiye
# 2. Arrival delay minutes > 0 hone chahiye
df_delay_base = df_silver.filter(
    (col("is_cancelled") == False) & (col("ArrDelayMinutes") > 0)
)

# Step 2: Group by year, month, airline_code

df_delay_cause_table = df_delay_base.groupBy(
    "flight_year",
    "flight_month",
    "airline_code"
).agg(
    # 1. Total arrival delay minutes
    round(
        sum("ArrDelayMinutes"),
        2
    ).alias("total_minutes_delayed"),

    # 2. Total weather delay minutes
    round(
        sum("WeatherDelay"),
        2
    ).alias("total_weather_delayed_minutes"),

    # 3. Total carrier delay minutes
    round(
        sum("CarrierDelay"),
        2
    ).alias("total_carrier_delayed_minutes"),

    # 4. Total security delay minutes
    round(
        sum("SecurityDelay"),
        2
    ).alias("total_security_delayed_minutes"),

    # 5. Total late aircraft delay minutes
    round(
        sum("LateAircraftDelay"),
        2
    ).alias("total_late_aircraft_delayed_minutes")
)


# Step 3: Percentage calculation for each delay cause
# Formula:
# (cause_delay_minutes / total_minutes_delayed) * 100

df_delay_cause_table = df_delay_cause_table.withColumn(
    "weather_delay_percentage",
    round(
        (col("total_weather_delayed_minutes") / col("total_minutes_delayed")) * 100,
        2
    )
).withColumn(
    "carrier_delay_percentage",
    round(
        (col("total_carrier_delayed_minutes") / col("total_minutes_delayed")) * 100,
        2
    )
).withColumn(
    "security_delay_percentage",
    round(
        (col("total_security_delayed_minutes") / col("total_minutes_delayed")) * 100,
        2
    )
).withColumn(
    "late_aircraft_delay_percentage",
    round(
        (col("total_late_aircraft_delayed_minutes") / col("total_minutes_delayed")) * 100,
        2
    )
)


# Step 4: Final display

# display(df_delay_cause_table)


# Step 5: Save as Delta Table
# Table name = delay_cause_table
df_delay_cause_table.write.format("delta").mode("overwrite").saveAsTable("delay_cause_table")